# Strict AOG v39 Template Quality Diagnostics

This notebook adds the missing template-level diagnostics: usage, simplicity, and representativeness. It compares v38 and v39 because v38 has the best validation accuracy while v39 has the cleanest structural audit.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'experiments' / 'aog_v19_v23').exists():
            return candidate
    return start

ROOT = find_repo_root()
V38 = ROOT / 'experiments/aog_v19_v23/runs/strict_aog_v38_template_quality'
V39 = ROOT / 'experiments/aog_v19_v23/runs/strict_aog_v39_template_quality'
print('ROOT:', ROOT)
print('v38 exists:', V38.exists())
print('v39 exists:', V39.exists())

## Class-Level Usage and Simplicity

`effective_templates_used` measures how many templates are actually used per class on validation examples. `mean_slots` and `mean_edges` measure simplicity. A good class should not need one very large template to cover most examples unless the class truly has one stable layout.


In [ ]:
v38c = pd.read_csv(V38 / 'strict_aog_v38_template_quality_by_class.csv')
v39c = pd.read_csv(V39 / 'strict_aog_v39_template_quality_by_class.csv')
cols = ['class_name','accuracy','effective_templates_used','max_usage_frac','mean_slots','mean_edges','mean_position_std','num_low_usage_templates','num_high_use_complex_templates','num_high_degree_templates']
display(Markdown('### v38'))
display(v38c[cols])
display(Markdown('### v39'))
display(v39c[cols])

## Template-Level Flags

Flags include low-use templates, high-use complex templates, high-degree templates, and rigid high-use layouts.


In [ ]:
v39t = pd.read_csv(V39 / 'strict_aog_v39_template_quality_by_template.csv')
flagged = v39t[v39t['quality_flags'].fillna('') != '']
display(flagged[['class_name','template_id','true_usage_frac','accuracy_when_true_template','num_slots','num_edges','max_degree','position_std_mean','quality_flags']])

## Visual Diagnostics

The dashboard shows template usage heatmaps, per-template accuracy, effective template counts, and usage vs slot complexity. The scatter plot shows edges-per-slot against slot position variance, with marker size proportional to usage.


In [ ]:
display(Markdown('### v39 dashboard'))
display(Image(filename=str(V39 / 'strict_aog_v39_template_quality_dashboard.png')))
display(Markdown('### v39 simplicity/flexibility scatter'))
display(Image(filename=str(V39 / 'strict_aog_v39_simplicity_flexibility_scatter.png')))
display(Markdown('### v38 dashboard'))
display(Image(filename=str(V38 / 'strict_aog_v38_template_quality_dashboard.png')))
display(Markdown('### v38 simplicity/flexibility scatter'))
display(Image(filename=str(V38 / 'strict_aog_v38_simplicity_flexibility_scatter.png')))

## Current Interpretation

The old audit was structural only. The new diagnostics show v39 is simpler than v38 in edge complexity and removes high-degree templates, but it still has representativeness problems: several low-use alternatives and a high-use complex aeroplane template. That suggests the next design should not merely reduce edges; it should simplify slot definitions and make template alternatives cover distinct layout modes.
